In [1]:
!pip install langchain  langchain-core  langchain-community  pypdf  pymupdf langchain_text_splitters

In [3]:
# Document Structure

from langchain_core.documents import Document

In [4]:
doc = Document(
    page_content = "this is a main text content I am using to create RAG",
    metadata = {
        "source" : "example.txt",
        "page" : 1,
        "author" : "saif ullah",
        "date_creted" : "2026-13-07"
        
    }
)
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'saif ullah', 'date_creted': '2026-13-07'}, page_content='this is a main text content I am using to create RAG')

In [6]:
# creating simple txt file
import os 
os.makedirs("../data/text_file.txt" , exist_ok=True)

In [13]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems"""
}

import os

os.makedirs("data/text_files", exist_ok=True)

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("sample text files created")

sample text files created


In [15]:
## TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt" , encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [17]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

# DirectoryLoader ek folder ke andar multiple files ko load karne ke liye use hota hai
dir_loader = DirectoryLoader(

    # Jis folder se files load karni hain
    "data/text_files",

    # Sirf .txt files ko find/load karega
    # ** ka matlab folder aur subfolders mein bhi search karega
    glob="**/*.txt",

    # Found text files ko read karne ke liye TextLoader use hoga
    loader_cls=TextLoader,

    # TextLoader ki additional settings
    # UTF-8 encoding ke saath files read hongi
    loader_kwargs={"encoding": "utf-8"},

    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming l

In [26]:
# for pdf loader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader , DirectoryLoader

dir_loader = DirectoryLoader(
    "data/pdfs",
    glob = "**/*.pdf",
    loader_cls = PyMuPDFLoader,
    # loader_kwargs = {'encoding':'utf-8'},
    show_progress = False
)
pdf_documnts = dir_loader.load()
pdf_documnts 


[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': '', 'creationdate': '2026-03-21T04:11:43+00:00', 'source': 'data\\pdfs\\reasearch1.pdf', 'file_path': 'data\\pdfs\\reasearch1.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2026-03-22T10:14:18+05:30', 'trapped': '', 'modDate': "D:20260322101418+05'30", 'creationDate': "D:20260321041143+00'00'", 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kai

In [27]:
type(pdf_documnts[0])

langchain_core.documents.base.Document

### RAG Pipeline Data loading to embeding 

In [9]:
import os 
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Lenovo\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    # Sab PDFs ke loaded documents ko store karne ke liye empty list
    all_documents = []

    # Given directory path ko Path object mein convert karta hai
    pdf_dir = Path(pdf_directory)

    # Directory aur uske subdirectories mein tamam .pdf files find karta hai
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    # Total milne wali PDF files ki quantity display karta hai
    print(f"Found {len(pdf_files)} PDF files to process")

    # Har PDF file ko one by one process karta hai
    for pdf_file in pdf_files:

        # Current PDF ka naam display karta hai
        print(f"\nProcessing: {pdf_file.name}")

        try:
            # Current PDF ko PyPDFLoader se load karta hai
            # str() Path object ko string path mein convert karta hai
            loader = PyPDFLoader(str(pdf_file))

            # PDF ke pages ko LangChain Document objects mein convert karta hai
            documents = loader.load()

            # Har page/document ki metadata update karta hai
            for doc in documents:

                # PDF file ka naam metadata mein save karta hai
                doc.metadata["source_file"] = pdf_file.name

                # File type metadata mein save karta hai
                doc.metadata["file_type"] = "pdf"

            # Current PDF ke documents ko main list mein add karta hai
            all_documents.extend(documents)

            # Current PDF mein kitne pages load hue woh display karta hai
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:

            # Agar kisi PDF ko load karne mein error aaye
            # to program completely stop nahi hoga
            print(f"  ✗ Error: {e}")

    # Sab PDFs ke total loaded pages/documents display karta hai
    print(f"\nTotal documents loaded: {len(all_documents)}")

    # Final documents list return karta hai
    return all_documents


# data folder ke andar tamam PDFs process karega
all_pdf_documents = process_all_pdfs("data")

c:\Users\Lenovo\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 2 PDF files to process

Processing: reasearch1.pdf
  ✓ Loaded 11 pages

Processing: research2.pdf
  ✓ Loaded 21 pages

Total documents loaded: 32


In [14]:
pdf_dir = Path("data/pdfs")
pdf_files = list(pdf_dir.glob("**/*.pdf"))
print(pdf_files[0].name)

reasearch1.pdf


In [ ]:
def process_all_pdfs(pdf_directory):
    all_pdf_documents = []
    
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Total no of pdfs = | {len(pdf_files)} |")
    
    print("Loading process continue ......")
    
    try:
        for pdf_file in pdf_files:
            print(f"{pdf_file.name} is loaded")
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
                
            all_pdf_documents.extend(documents)
            print(f"No of pages {len(documents)} in dpf = {pdf_file.name}")
    except Exception as e:
        print(f"Error : | {e} |") 
        
    print(f"Toal no of pages in all pdfs = {len(all_pdf_documents)}")
    return all_pdf_documents

all_pdf_documents = process_all_pdfs("data")

Total no of pdfs = | 2 |
Loading process continue ......
reasearch1.pdf is loaded
No of pages 11 in dpf = reasearch1.pdf
research2.pdf is loaded
No of pages 21 in dpf = research2.pdf
Toal no of pages in all pdfs = 32


In [18]:
# all_pdf_documents


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Documents ko small chunks mein split karne ke liye function
def split_documents(documents, chunk_size=500, chunk_overlap=50):

    # RecursiveCharacterTextSplitter ka object create kar rahe hain
    text_splitter = RecursiveCharacterTextSplitter(

        # Har chunk ka maximum size
        chunk_size=chunk_size,

        # Previous chunk ke kuch characters next chunk mein repeat honge
        # Isse context maintain rehta hai
        chunk_overlap=chunk_overlap,

        # Text ki length calculate karne ke liye Python ka len() function
        length_function=len,

        # Text ko split karne ke liye separators
        # Pehle paragraph, phir line, phir space, phir character level
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    # Check kar rahe hain ke chunks successfully create hue ya nahi
    if split_docs:
        print("\nExample chunk:")

        print(
            f"Content: {split_docs[0].page_content[:100]}..."
        )
        print(
            f"Metadata: {split_docs[0].metadata}"
        )
    return split_docs

In [3]:
chunks = split_documents(all_pdf_documents)

Split 32 documents into 320 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brai...
Metadata: {'producer': 'pdfcpu v0.9.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-03-21T04:11:43+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing w

In [27]:
# !pip install sentence-transformers faiss-cpu chromadb

### Embedings creations

In [4]:

from sentence_transformers import SentenceTransformer

In [5]:
class EmbedingManager:
    def __init__(self , model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print(f"Loading Embeding Model {self.model_name} .........")
        self.model = SentenceTransformer(self.model_name)
        print(f" Model is loaded successfully  \n Embeding Dimensions = {self.model.get_sentence_embedding_dimension()}")
        
    def generate_embedings(self , chunks):
        embedings = self.model.encode(chunks , show_progress_bar = True)
        if embedings:
            print("Embeding created successfully .....")
            print(f"Embeding shapes = {embedings.shape}")
            
        return embedings

In [8]:
embeding_manager = EmbedingManager()

Loading Embeding Model all-MiniLM-L6-v2 .........


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

### Vectorestore

In [7]:
import chromadb
import uuid
import os

In [20]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [21]:
vectore_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 640


In [ ]:
texts = [doc.page_content for doc in chunks]

embedings = embeding_manager.generate_embedings(texts)

vector_store.add_documents(documents , embedings)
